In [2]:
import pandas as pd 
import numpy as np 
import os

# 1. Klienti
Centrālā tabula ir klienti, nofiltrētie pēc Listed no 01.01.2022. līdz 31.12.2022.

In [31]:
# Automātiski atjaunojams fails ar datiem par visiem klientiem 
clients_path = r"G:\GSCLV-FIN PUBLIC\-=DATI=-\Report Automatization TheTable\The Table.csv"
clients_df = pd.read_csv(
    clients_path,
    sep=';',
    engine='python',
    on_bad_lines='warn',
    encoding='utf-8',
    encoding_errors='replace'
)

In [32]:
# Atstāt noteiktas kolonnas clients_df:
clients_clean = clients_df[['File', 
                            'SSN',
                            'Client number', # client ID
                            'Legal start', # for legal_flag
                            'Principal', # debt amount
                            'Agree.Date', # agreement date
                            'Termin.date', # agreement termination date
                            'Listed', # purchase date
                            'LastPaym.', # last payment date
                            'City', #  city/region
                            'User 1', # service
                            'Home', # is_phone flag
                            'Cell',
                            'Email', # is_email flag
                            'ZIP', # region
                            'Closed', # date of paid in full 
                            'Type', # private or business
                            'Status' 
                            ]]
clients_clean['Listed'] = pd.to_datetime(clients_clean['Listed'], errors='coerce', format='%d/%m/%Y') 
# Nofiltrējam rindas, kur Listed ir no 2022. gada 1. janvāra līdz 2022. gada 31. decembrim un Type = PI
# Nofiltrējam testa klientus, kuriem Client number nav 200, 2001, 2002, 20000
clients_clean = clients_clean[
    (clients_clean['Listed'] >= '2022-01-01') & (clients_clean['Listed'] < '2023-01-01') & (clients_clean['Type'] == 'PI') &
    (clients_clean['Client number'].isin([200, 2001, 2002, 20000, 20110, 20136]) == False)]
# 200, 2001, 2002, 20000 ir testa portfeļi, 20110 ir secured
# 20136 sastāv tikai no juridiskām personām, tāpēc to izslēdzam no saraksta.
# Saglabāt csv failu:
#clients_clean.to_csv(r"G:\GSCLV-FIN\Current month BI Reports\Client profile//2026//Valuation data\clients_listed2022_pi.csv", index=False)
clients_clean.describe(include='all')   

,File,SSN,Client number,Legal start,Principal,Agree.Date,Termin.date,Listed,LastPaym.,City,User 1,Home,Cell,Email,ZIP,Closed,Type,Status
count,14213.000000,14213,14213.000000,9316,14213.000000,14180,13698,14213,9971,14200,13494,11309,8445,12076,14204,4942,14213,13391
unique,NaN,10673,NaN,281,NaN,2640,2638,NaN,1175,2009,69,8512,6520,9044,956,1086,1,21
top,NaN,150282-11363,NaN,Legal 04/25,NaN,18/05/2021,18/05/2021,NaN,01/01/1900,RĪGA,Banknote,37127890445,37127890445,lana.radzina@dhl.com,LV-3401,19/10/2022,PI,ZTI
freq,NaN,10,NaN,757,NaN,58,57,NaN,312,3652,4710,10,10,10,294,97,14213,5434
mean,33127.858158,NaN,20112.610286,NaN,1052.072493,NaN,NaN,2022-08-20 18:29:18.333919,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,24896.000000,NaN,20059.000000,NaN,0.000000,NaN,NaN,2022-01-21 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,29217.000000,NaN,20089.000000,NaN,162.010000,NaN,NaN,2022-05-15 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,32955.000000,NaN,20116.000000,NaN,542.510000,NaN,NaN,2022-09-15 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,37461.000000,NaN,20142.000000,NaN,1192.530000,NaN,NaN,2022-12-13 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,41054.000000,NaN,20153.000000,NaN,67048.220000,NaN,NaN,2022-12-30 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
clients_cl_df = clients_clean.copy()

clients_cl_df['Agree.Date'] = pd.to_datetime(
    clients_cl_df['Agree.Date'],
    errors='coerce',
    format='%d/%m/%Y'
)

clients_cl_df['Termin.date'] = pd.to_datetime(
    clients_cl_df['Termin.date'],
    errors='coerce',
    format='%d/%m/%Y'
)

clients_cl_df['LastPaym.'] = pd.to_datetime(
    clients_cl_df['LastPaym.'],
    errors='coerce',
    format='%d/%m/%Y'
)

clients_cl_df['Closed'] = pd.to_datetime(
    clients_cl_df['Closed'],
    errors='coerce',
    format='%d/%m/%Y'
)

#Pievienot kolonnu vai ir Legal start datums, lai varētu izveidot legal_flag: legal vai non-legal:  
clients_cl_df['legal_flag'] = np.where(clients_cl_df['Legal start'].notna(), 'legal', 'non-legal')

# Pievienot kolonnu, contact_flag: ja ir Home, Cell vai Email, tad contact_flag ir contactable, citādi non-contactable:
clients_cl_df['contact_flag'] = np.where((clients_cl_df['Home'].notna()) | (clients_cl_df['Cell'].notna()) | (clients_cl_df['Email'].notna()), 'contactable', 'non-contactable')

# Paid or not flag: ja Status ir PIF, tad paid_flag is paid, citādi unpaid:
clients_cl_df['paid_flag'] = np.where(clients_cl_df['Status'] == 'PIF', 'paid', 'unpaid')

# Izveidojam kolonnu ar lietu skaitu katram klientam:
clients_cl_df['number_of_cases_per_ssn'] = clients_cl_df.groupby('SSN')['File'].transform('count')

#Izņemam kolonnas, kuras vairs nav nepieciešamas: Legal start, Home, Cell, Email, City, Type, Status, SSN:
columns_to_drop = ['Legal start', 'Home', 'Cell', 'Email', 'City', 'Type', 'Status', 'SSN']
clients_cl_df = clients_cl_df.drop(columns=columns_to_drop)

In [9]:
clients_cl_df.info()

<class 'pandas.DataFrame'>
Index: 14213 entries, 5 to 60013
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   File                     14213 non-null  int64         
 1   Client number            14213 non-null  int64         
 2   Principal                14213 non-null  float64       
 3   Agree.Date               14180 non-null  datetime64[us]
 4   Termin.date              13698 non-null  datetime64[us]
 5   Listed                   14213 non-null  datetime64[us]
 6   LastPaym.                9971 non-null   datetime64[us]
 7   User 1                   13494 non-null  str           
 8   ZIP                      14204 non-null  str           
 9   Closed                   4926 non-null   datetime64[us]
 10  legal_flag               14213 non-null  str           
 11  contact_flag             14213 non-null  str           
 12  paid_flag                14213 non-null  str    

In [34]:
clients_cl_df['File'].duplicated().sum()

np.int64(0)

# 2. Dzimums un vecums
Ielādējam iepriekš sagatavotu failu ar datiem par klientu dzimumu un vecuma grupu. Ja vecumu un dzimumu nebija iespējams noteikt, lietai ir legal/new SSN vērtība kolonnās Age_group un Gender.

In [35]:
# R ģenerēts fails ar klientu dzimumu un vecumu:
age_gender_path = r"G:\GSCLV-FIN\Current month BI Reports\Client profile\2026\Valuation data\clients_age_gender_df.csv"
age_gender_df = pd.read_csv(age_gender_path, sep=';')
# Atstājam tikai kolonnas File, Age_group, Gender:
age_gender_cl_df = age_gender_df[['File', 'Age_group', 'Gender']]

In [36]:
age_gender_cl_df['Age_group'].value_counts(dropna=False)

Age_group
31-50            7248
<30              3413
51-70            2793
legal/new SSN     721
71-90             690
90+                52
Name: count, dtype: int64

In [11]:
age_gender_cl_df['Gender'].value_counts(dropna=False)

Gender
male             8337
female           5805
legal/new SSN     775
Name: count, dtype: int64

In [37]:
age_gender_cl_df['File'].duplicated().sum()

np.int64(0)

# 3. Kavējuma datums
Kavējuma datums ir būtiska vērtība, kas ļauj saprast, kad klients pirmo reizi ir nokavējis maksājumu.

In [38]:
# kavējuma datumu fails 
delay_path = r"G:\GSCLV-FIN\Current month BI Reports\Client profile\2026\Valuation data\Kavejuma_datums.xlsx"
delay_df = pd.read_excel(delay_path)

In [39]:
delay_df['File'].duplicated().sum()

np.int64(0)

# 4. Reģioni


In [40]:
regions_path = r"G:\GSCLV-FIN\Current month BI Reports\Client profile\2026\Valuation data\regions.csv"
regions_df = pd.read_csv(
    regions_path,
    sep=';',
    engine='python',
    encoding='cp1257',
    on_bad_lines='warn'
)

# Pārdēvēt Pasta Indekss kolonnu nosaukumu uz "ZIP", lai skaidri norādītu, ka tā attiecas uz pasta indeksu unificēšanai:
regions_df = regions_df.rename(columns={'Pasta Indekss': 'ZIP'})

regions_df["region_len"] = regions_df["Reģions"].str.len()

regions_df_sorted = regions_df.sort_values(["ZIP", "region_len"])
regions_df_dedup = regions_df_sorted.drop_duplicates(subset="ZIP", keep="first").drop(columns="region_len")

# Pārbaudām, vai nav ZIP dublikātu:
duplicate_zips = regions_df_dedup['ZIP'].duplicated().sum()
print(f"Number of duplicate 'ZIP' entries: {duplicate_zips}")

Number of duplicate 'ZIP' entries: 0


# 5. Datu Apvienošana

In [41]:
# Apvienojam datus pa 'File' kolonnu, lai iegūtu vienu kopīgu datu kopu:
combined_df = pd.merge(clients_cl_df, age_gender_cl_df, on='File', how='left')
combined_df2 = pd.merge(combined_df, delay_df, on='File', how='left')
combined_df3 = pd.merge(combined_df2, regions_df_dedup, on='ZIP', how='left')

# Izdzēšam nevajadzīgos failus, lai atbrīvotu atmiņu:
del clients_df, clients_clean, clients_cl_df, age_gender_cl_df, delay_df, regions_df_dedup

In [42]:
# Pārbaudām, vai nav File dublikātu:
duplicate_files = combined_df3['File'].duplicated().sum()
print(f"Number of duplicate 'File' entries: {duplicate_files}")

Number of duplicate 'File' entries: 0


In [43]:
# Pārsaucam kolonnu Kavējuma datums uz Delay_date, lai skaidri norādītu, ka tā attiecas uz kavējuma datumu unificēšanai:
clients_final_df = combined_df3.rename(columns={'Kavējuma datums': 'Delay_date'})

# Ja Delay_date ir 1900-01-01, tad aizstāt ar NaN, jo tas nozīmē, ka kavējuma datums nav zināms:
clients_final_df['Delay_date'] = clients_final_df['Delay_date'].replace(pd.Timestamp('1900-01-01'), pd.NaT)

# Izveidojam jaunas kolonnas
# 1. No kavējuma sākuma līdz cesijai. Tas raksturo, cik “vecs” parāds bija pirkšanas brīdī. Vērtība ir dienās, un to iegūst, atņemot kavējuma datumu no Listed datuma un piešķirot pareizo datu tipu:
clients_final_df['debt_age_at_purchase_days'] = (clients_final_df['Listed'] - clients_final_df['Delay_date']).dt.days

# 2. No cesijas / pirkšanas līdz nomaksai. Tas rāda, cik ilgs laiks pagāja no portfeļa iegādes līdz slēgšanai / nomaksai.
clients_final_df['days_from_listed_to_closed'] = (clients_final_df['Closed'] - clients_final_df['Listed']).dt.days

# 3. No pēdējā maksājuma līdz cesijai. Tas rāda, cik “svaigs” vai “auksts” bija klients pirms pirkšanas.
clients_final_df['days_from_lastpaym_to_listed'] = (clients_final_df['Listed'] - clients_final_df['LastPaym.']).dt.days


In [44]:
# Izņemam nevajadzīgās kolonnas, lai atbrīvotu atmiņu:
columns_to_drop_final = ['Agree.Date', 'Termin.date', 'Delay_date', 'Listed', 'Closed', 'LastPaym.', 'Portfeļa nosaukums', 'User 1']
clients_final_df = clients_final_df.drop(columns=columns_to_drop_final)

# 5. Maksājumi
Maksājumi ļaus izvērtēt klienta uzvedību laika gaitā. 

In [45]:
transactions_path = r"G:\GSCLV-FIN\Current month BI Reports\Client profile\2026\Valuation data\transactions_101.csv"
transactions_df = pd.read_csv(transactions_path, sep=';', low_memory=False)

In [46]:
transactions_cl_df = transactions_df[['UID', 'File', 'Client', 'Payment date', 'To us']]
transactions_cl_df['Payment date'] = pd.to_datetime(transactions_cl_df['Payment date'], errors='coerce', format='%d/%m/%Y')
#nofiltrēt tikai maksājumus ar summu To us >0 un Client nav vienāds ar 20110:
transactions_cl_df = transactions_cl_df[(transactions_cl_df['To us'] > 0) & (transactions_cl_df['Client'].isin([200, 2001, 2002, 20000, 20110, 20136]) == False)]
# pārdēvēt To us kolonnas nosaukumu uz Payment_amount
transactions_cl_df = transactions_cl_df.rename(columns={'To us': 'Payment_amount', 'UID': 'transaction_id'})
#mainīt datu tipu kolonnai transaction_id uz integer: 
transactions_cl_df['transaction_id'] = transactions_cl_df['transaction_id'].astype(int)
del transactions_df

# 6. Aktivitātes

In [4]:
# Visas aktivitātes, ar V.U., p.k. un lietu numuriem
#events_path = r"G:\GSCLV-FIN PUBLIC\Current month BI Reports\Client profile\2026\Valuation data\contacts.csv"
events_path = r"G:\GSCLV-FIN PUBLIC\Current month BI Reports\Client profile\2026\Valuation data\events_cl_df.csv"
events_df = pd.read_csv(events_path, sep=',', low_memory=False)


In [ ]:
# Cikls, kas atkārto darbības ar katru failu mapē. Nolasām datus un atstājam tikai noteiktas kolonnas: File, Client, Type, Collector, Relation, Created date, Done date, Due date, Description:

folder_path = r"G:\GSCLV-FIN PUBLIC\Current month BI Reports\Client profile\2026\Valuation data\contacts"
events_df = pd.DataFrame()  # Izveidojam tukšu DataFrame, lai saglabātu visus datus
for file_name in os.listdir(folder_path):
    if file_name.endswith('.csv'):
        file_path = os.path.join(folder_path, file_name)
        temp_df = pd.read_csv(file_path, sep=';', low_memory=False)
        columns_to_keep = ['File', 'Client', 'Type', 'Collector', 'Assigned by', 'Created date', 'Done date', 'Due date', 'Description']
        temp_df = temp_df[columns_to_keep]
        events_df = pd.concat([events_df, temp_df], ignore_index=True)


In [ ]:
events_cl_df = events_df.copy()
# Izņemam rindas ar trūkstošiem datiem File un Client kolonnās, jo tās ir būtiskas apvienošanai ar citiem datiem:
events_cl_df = events_cl_df.dropna(subset=['File', 'Client'])

# File un Client kolonnas datu tipam jābūt integer, lai varētu apvienot ar citiem datiem:
events_cl_df['File'] = events_cl_df['File'].astype(int)
events_cl_df['Client'] = events_cl_df['Client'].astype(int)

# Datumu kolonnām jābūt datetime tipam, lai varētu veikt datuma aprēķinus:
events_cl_df['Created date'] = pd.to_datetime(events_cl_df['Created date'], errors='coerce', format='%Y-%m-%d')
events_cl_df['Done date'] = pd.to_datetime(events_cl_df['Done date'], errors='coerce', format='%Y-%m-%d')
events_cl_df['Due date'] = pd.to_datetime(events_cl_df['Due date'], errors='coerce', format='%Y-%m-%d')
events_cl_df = events_cl_df[events_cl_df['Client'].isin([200, 2001, 2002, 20000, 20110, 20136]) == False]
# Atstājam tikai rindas ar Client >= 20059 un <= 20153
events_cl_df = events_cl_df[(events_cl_df['Client'] >= 20059) & (events_cl_df['Client'] <= 20153)]

del events_df
events_cl_df.to_csv(r"G:\GSCLV-FIN\Current month BI Reports\Client profile\2026\Valuation data\events_cl_filtered_df.csv", index=False)

In [24]:
# Relācijas tabula kontaktu unificēšanai
relation_table = r"G:\GSCLV-FIN\Current month BI Reports\Client profile\2026\Valuation data\relation_table.csv"
relation_df = pd.read_csv(
    relation_table,
    sep=';',
    engine='python',
    encoding='utf-8',
    on_bad_lines='warn'
)

In [51]:
# Savienojam events_cl_df ar relation_df, lai iegūtu vienotu kontaktu unificēšanas tabulu, pamatojoties uz 'Type' un 'Collector' kolonnām, izmantojot left join, lai saglabātu visus ierakstus no events_cl_df un pievienotu atbilstošās vērtības no relation_df:
final_relation_df = pd.merge(events_cl_df2, relation_df, on=['Type', 'Collector'], how='left')

In [52]:
#Pārbaudīt, vai ir kādi null vērtību rezultāti kplonnā Relation pēc apvienošanas:
null_relation_count = final_relation_df['Relation'].isnull().sum()
print(f"Number of null values in 'Relation' column: {null_relation_count}")

Number of null values in 'Relation' column: 44083


In [53]:
# Ja Type = "Email", bet Relation ir NA, tad Relation ir "EMAIL":
final_relation_df.loc[(final_relation_df['Type'] == 'Email') & (final_relation_df['Relation'].isna()), 'Relation'] = 'EMAIL'
# # Ja Type = Email, Collector nav NA, tad Relation ir "EMAIL":
final_relation_df.loc[(final_relation_df['Type'] == 'Email') & (final_relation_df['Collector'].notna()), 'Relation'] = 'EMAIL'
# # Ja Type = Review, bet Collector ir NA, tad Relation ir "remove":
final_relation_df.loc[(final_relation_df['Type'] == 'Review') & (final_relation_df['Collector'].isna()), 'Relation'] = 'remove'
# # Ja Type = Close, Relation ir "remove":
final_relation_df.loc[final_relation_df['Type'] == 'Close', 'Relation'] = 'remove'

In [54]:
# Ja kolonnas Description vērtība satur "dialer", tad Relation ir "DIALER":
final_relation_df.loc[final_relation_df['Description'].str.contains('dialer', case=False, na=False), 'Relation'] = 'DIALER'

In [29]:
# Nofiltrēt tikai rindas, kur Relation ir NA:
final_df_na_relation = final_relation_df[final_relation_df['Relation'].isna()]
# Rindu skaits, kur Relation ir NA:
na_relation_count = final_df_na_relation.shape[0]
print(f"Number of rows with NA in 'Relation' column: {na_relation_count}")

Number of rows with NA in 'Relation' column: 7774


In [30]:
# Saglabāt Excel formātā:
final_df_na_relation.to_excel("G:\\GSCLV-FIN\\Current month BI Reports\\Client profile\\2026\\Valuation data\\missing_relation_values_for_manual_check.xlsx", index=False)

In [ ]:
# Relation vērtību sadalījums:
# relation_counts = final_relation_df['Relation'].value_counts(dropna=False)
# print("Relation value counts:")
# print(relation_counts)

In [55]:
# Izņemam rindas, kur Relation ir "remove", jo tās nav derīgas analīzei:
final_relation_clean_df = final_relation_df[final_relation_df['Relation'] != 'remove']
# Atstāt tikai kolonnas, kas nepieciešamas turpmākai analīzei:
columns_to_keep = ['File', 'Client', 'Type', 'Collector', 'Relation', 'Created date', 'Done date', 'Due date', 'Description']
final_relation_clean_df = final_relation_clean_df[columns_to_keep]
nas_df = final_relation_clean_df[final_relation_clean_df['Relation'].isna()]
final_relation_clean_df = final_relation_clean_df[final_relation_clean_df['Relation'].notna() & final_relation_clean_df['File'].notna()]
# Client un File jābūt integer, lai varētu apvienot ar citiem datiem:
final_relation_clean_df['Client'] = final_relation_clean_df['Client'].astype(int)
final_relation_clean_df['File'] = final_relation_clean_df['File'].astype(int)
columns_to_drop_final = ['Collector', 'Type', 'Description']
final_relation_clean_df = final_relation_clean_df.drop(columns=columns_to_drop_final)
#pārdēvēt Relation kolonnu uz Type, lai skaidri norādītu, ka tā attiecas uz kontaktu veidu:
final_relation_clean_df = final_relation_clean_df.rename(columns={'Relation': 'Type'})

Maksājumi: No 2022.gada 1.janvāra (Closed)

# Kontaktēšanas datuma izveide un lieko kolonnu noņemšana

Šajā posmā tiek izveidota vienota kontaktēšanas datuma kolonna un no datu kopas izņemtas vairākas kolonnas, kuras turpmākajā analīzē vairs nav nepieciešamas.

## Kolonnas Contact_date izveide

Lai katrai aktivitātei būtu viens galvenais datums, kas raksturo kontaktēšanas brīdi, tiek izveidota jauna kolonna Contact_date.

Tās izveides loģika ir šāda:

- ja kolonnā Done date ir aizpildīta vērtība, par kontaktēšanas datumu tiek izmantots šis datums;
- ja Done date nav pieejams, tad tiek izmantots Due date.

Šāda pieeja ļauj iegūt vienotu datuma lauku arī tajos gadījumos, kad aktivitātes izpildes datums nav reģistrēts, bet ir pieejams plānotais vai noteiktais izpildes termiņš.

Kolonna Contact_date vēlāk kalpo kā galvenais laika atskaites punkts:

- aktivitāšu secību veidošanai,
- kontaktu sakārtošanai hronoloģiskā secībā,
- dienu starpību aprēķiniem,
- maksājuma reakcijas logu analīzei pēc kontakta.
- Lieko kolonnu noņemšana

Pēc kolonnas Contact_date izveides no tabulas tiek izņemtas vairākas kolonnas, kuras vairs nav nepieciešamas, jo:

- to informācija jau ir apvienota jaunajā kolonnā,
- tās nav būtiskas turpmākajai secību analīzei,
- vai arī tās tiek uzskatītas par tehnisku vai lieku informāciju šajā datu kopas versijā.

Tiek dzēstas šādas kolonnas:

- Done date — jo tās vērtība jau ir izmantota Contact_date izveidē,
- Due date — jo arī šī kolonna jau ir izmantota Contact_date loģikā,
- Created date — jo turpmākajā analīzē galvenais kontakta laika punkts ir Contact_date,
- ZIP — jo reģiona informācija jau ir pievienota citā veidā un pasta indekss šajā posmā vairs nav nepieciešams,
- Collector — jo pēc aktivitāšu standartizācijas šī informācija vairs nav būtiska konkrētajai secību analīzes versijai.

In [56]:
# Izveidojam kolonnu Contact_date. Ja ir Done date, tad Contact_date ir Done date, citādi Contact_date ir Due date:
final_relation_clean_df['Contact_date'] = np.where(final_relation_clean_df['Done date'].notna(), final_relation_clean_df['Done date'], final_relation_clean_df['Due date'])
final_events_clean_df = final_relation_clean_df.drop(columns=['Done date', 'Due date', 'Created date'])
del final_relation_df, final_relation_clean_df

Portfeļu sastāva pārbaude


In [57]:
unique_clients = final_events_clean_df['Client'].unique()
unique_client_numbers = clients_final_df['Client number'].unique()
# Print unique_clients that are not in unique_client_numbers
unique_clients_not_in_numbers = set(unique_clients) - set(unique_client_numbers)
print("Unique clients that are not in unique client numbers:")
print(unique_clients_not_in_numbers)
#20136 tikai juridiskās personas


Unique clients that are not in unique client numbers:
set()


In [58]:
# saglabāt csv failu:
#clients_final_df.to_csv(r"G:\GSCLV-FIN\Current month BI Reports\Client profile\2026\Valuation data\clients_final_df.csv", index=False, sep=';')
# transactions_cl_df.to_csv(r"G:\GSCLV-FIN\Current month BI Reports\Client profile\2026\Valuation data\transactions_cl_df.csv", index=False, sep=';')
final_events_clean_df.to_csv(r"G:\GSCLV-FIN\Current month BI Reports\Client profile\2026\Valuation data\final_events_clean_df.csv", index=False, sep=';')
